# 图论指标 + ROI 连接性 组间统计分析

对 ADHD 单纯组 (adhd)、ADHD 共患阅读困难组 (com)、正常发展组 (td) 进行连接性指标的组间统计比较。

**分析内容:**
- 全脑图论指标: 聚类系数、路径长度、小世界系数、平均连接强度 (4 指标 × 5 频段 = 20 检验)
- ROI 连接性 (wPLI): Paper ROIs + Exploratory ROIs (9 ROI × 5 频段 = 45 检验)
- 合计 65 检验，统一 FDR 校正

**输出目录:** `reports/comparison/connectivity/`

## 1. 导入和配置

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))
from stats_utils import *

setup_plotting()

# 输出路径
OUTPUT_PATH = BASE_PATH / 'reports' / 'comparison' / 'connectivity'
OUTPUT_PATH, FIGURES_PATH = ensure_output_dirs(OUTPUT_PATH)

print(f'输出目录: {OUTPUT_PATH}')
print(f'图片目录: {FIGURES_PATH}')

输出目录: d:\LYW\REST_COM\reports\comparison\connectivity
图片目录: d:\LYW\REST_COM\reports\comparison\connectivity\figures


## 2. 数据加载 — 全脑连接性

In [2]:
import pandas as pd

def load_connectivity_data() -> pd.DataFrame:
    """加载三组连接性数据 (small_world_metrics.csv)"""
    dfs = []
    for g in GROUPS:
        csv_path = CONN_PATHS[g] / 'small_world_metrics.csv'
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            df['group'] = g
            dfs.append(df)
            print(f'  {GROUP_LABELS[g]}: {df["subject_id"].nunique()} 被试, {len(df)} 行')
        else:
            print(f'  ⚠ 未找到: {csv_path}')
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)


print('--- 加载连接性数据 ---')
conn_df = load_connectivity_data()
print(f'\n连接性数据: {conn_df.shape}')
conn_df.head()

--- 加载连接性数据 ---
  ADHD单纯组: 12 被试, 60 行
  ADHD共患阅读困难组: 14 被试, 70 行
  正常发展组: 8 被试, 40 行

连接性数据: (170, 7)


,subject_id,group,freq_band,clustering,path_length,sigma,avg_connectivity
0,18,adhd,delta,0.456371,0.283548,0.292213,0.290198
1,29,adhd,delta,0.455245,0.288406,0.286582,0.304734
2,31,adhd,delta,0.372534,0.254127,0.266147,0.264366
3,37,adhd,delta,0.392658,0.229922,0.310057,0.248486
4,38,adhd,delta,0.382433,0.258290,0.268816,0.278405


## 3. 数据加载 — ROI 连接性

从三组 `connectivity_results.npz` 中提取 ROI 内 / ROI 间平均 wPLI。

**Paper ROIs (论文驱动):**
- Prefrontal Network: Fp1, Fp2, AF3, AF4, F3, F4, F7, F8 (8 channels, ADHD 相关)
- Left POT: P3, P7, TP7, CP3, CP5, PO3 (6 channels, 阅读困难相关)

**Exploratory ROIs:** frontal_L / frontal_R / central_L / central_R / posterior_L / posterior_R

In [3]:
# ROI 定义
PAPER_ROIS = {
    'prefrontal': ['Fp1', 'Fp2', 'AF3', 'AF4', 'F3', 'F4', 'F7', 'F8'],
    'left_pot':   ['P3', 'P7', 'TP7', 'CP3', 'CP5', 'PO3'],
}

EXPLORATORY_ROIS = {
    'frontal_L':   ['Fp1', 'AF3', 'AF7', 'F1', 'F3', 'F5', 'F7'],
    'frontal_R':   ['Fp2', 'AF4', 'AF8', 'F2', 'F4', 'F6', 'F8'],
    'central_L':   ['FC1', 'FC3', 'FC5', 'C1', 'C3', 'C5', 'CP1', 'CP3', 'CP5'],
    'central_R':   ['FC2', 'FC4', 'FC6', 'C2', 'C4', 'C6', 'CP2', 'CP4', 'CP6'],
    'posterior_L':  ['P1', 'P3', 'P5', 'P7', 'PO3', 'PO7', 'O1'],
    'posterior_R':  ['P2', 'P4', 'P6', 'P8', 'PO4', 'PO8', 'O2'],
}

ALL_ROIS = {**PAPER_ROIS, **EXPLORATORY_ROIS}

print(f'Paper ROIs: {list(PAPER_ROIS.keys())}')
print(f'Exploratory ROIs: {list(EXPLORATORY_ROIS.keys())}')
print(f'总计 {len(ALL_ROIS)} 个 ROI')

Paper ROIs: ['prefrontal', 'left_pot']
Exploratory ROIs: ['frontal_L', 'frontal_R', 'central_L', 'central_R', 'posterior_L', 'posterior_R']
总计 8 个 ROI


In [4]:
import numpy as np

def get_channel_indices(channel_names: list, roi_channels: list) -> list:
    """获取 ROI 通道在完整通道列表中的索引"""
    indices = []
    for ch in roi_channels:
        if ch in channel_names:
            indices.append(channel_names.index(ch))
    return indices

def compute_within_roi_wpli(wpli_matrix: np.ndarray, roi_indices: list) -> float:
    """计算 ROI 内平均 wPLI (上三角均值)"""
    sub_matrix = wpli_matrix[np.ix_(roi_indices, roi_indices)]
    triu_idx = np.triu_indices(len(roi_indices), k=1)
    return float(np.mean(sub_matrix[triu_idx]))

def compute_between_roi_wpli(wpli_matrix: np.ndarray, roi1_indices: list, roi2_indices: list) -> float:
    """计算两个 ROI 之间的平均 wPLI"""
    sub_matrix = wpli_matrix[np.ix_(roi1_indices, roi2_indices)]
    return float(np.mean(sub_matrix))

def load_roi_connectivity() -> pd.DataFrame:
    """从三组 npz 文件提取 ROI 连接性指标, 返回长格式 DataFrame"""
    rows = []
    for g in GROUPS:
        npz_path = CONN_PATHS[g] / 'connectivity_results.npz'
        if not npz_path.exists():
            print(f'  ⚠ 未找到: {npz_path}')
            continue
        data = np.load(npz_path, allow_pickle=True)
        channel_names = list(data['channel_names'])
        subject_ids = list(data['subject_ids'])

        roi_indices = {}
        for roi_name, roi_chs in ALL_ROIS.items():
            idx = get_channel_indices(channel_names, roi_chs)
            if len(idx) >= 2:
                roi_indices[roi_name] = idx
            else:
                print(f'  ⚠ ROI {roi_name} 通道不足 ({len(idx)}/{len(roi_chs)}), 跳过')

        for band in FREQ_BAND_KEYS:
            if band not in data:
                continue
            matrices = data[band]
            for i, sub_id in enumerate(subject_ids):
                mat = matrices[i]
                # Within-ROI
                for roi_name, idx in roi_indices.items():
                    val = compute_within_roi_wpli(mat, idx)
                    rows.append({
                        'subject_id': sub_id, 'group': g, 'freq_band': band,
                        'roi_metric': f'within_{roi_name}', 'roi_type': 'within', 'value': val
                    })
                # Between-ROI (Paper ROIs only)
                paper_roi_names = [r for r in PAPER_ROIS if r in roi_indices]
                if len(paper_roi_names) == 2:
                    r1, r2 = paper_roi_names
                    val = compute_between_roi_wpli(mat, roi_indices[r1], roi_indices[r2])
                    rows.append({
                        'subject_id': sub_id, 'group': g, 'freq_band': band,
                        'roi_metric': f'between_{r1}_{r2}', 'roi_type': 'between', 'value': val
                    })

        print(f'  {GROUP_LABELS[g]}: {len(subject_ids)} 被试, {len(roi_indices)} ROIs')

    return pd.DataFrame(rows)


print('--- 加载 ROI 连接性数据 ---')
roi_df = load_roi_connectivity()
print(f'\nROI 数据: {roi_df.shape}')
print(f'ROI 指标: {roi_df["roi_metric"].nunique()} 个')
print(roi_df['roi_metric'].unique())

--- 加载 ROI 连接性数据 ---
  ADHD单纯组: 12 被试, 8 ROIs
  ADHD共患阅读困难组: 14 被试, 8 ROIs
  正常发展组: 8 被试, 8 ROIs

ROI 数据: (1530, 6)
ROI 指标: 9 个
['within_prefrontal' 'within_left_pot' 'within_frontal_L'
 'within_frontal_R' 'within_central_L' 'within_central_R'
 'within_posterior_L' 'within_posterior_R' 'between_prefrontal_left_pot']


## 4. Omnibus 检验 — 全脑连接性

4 指标 × 5 频段 = 20 检验

In [5]:
print('=' * 60)
print('全脑连接性 Omnibus 检验')
print('=' * 60)

conn_results = []
if not conn_df.empty:
    for metric in CONN_METRICS:
        if metric not in conn_df.columns:
            continue
        for band in FREQ_BAND_KEYS:
            sub_df = conn_df[conn_df['freq_band'] == band] if 'freq_band' in conn_df.columns else conn_df
            if sub_df.empty:
                continue
            res = run_analysis_for_metric(sub_df, metric, f'conn_{metric}', freq_band=band)
            if res:
                conn_results.append(res)
                sig_mark = '***' if res['p'] < 0.001 else '**' if res['p'] < 0.01 else '*' if res['p'] < 0.05 else ''
                print(f"  {metric:<20s} | {band:<6s} | {res['test']:<16s} | "
                      f"p={res['p']:.4f} {sig_mark:>3s} | {res['es_name']}={res['effect_size']:.3f}")

print(f'\n全脑连接性检验总数: {len(conn_results)}')

全脑连接性 Omnibus 检验
  clustering           | delta  | ANOVA            | p=0.5825     | η²=0.034
  clustering           | theta  | ANOVA            | p=0.1947     | η²=0.100
  clustering           | alpha  | ANOVA            | p=0.7176     | η²=0.021
  clustering           | beta   | ANOVA            | p=0.4809     | η²=0.046
  clustering           | gamma  | ANOVA            | p=0.6018     | η²=0.032
  path_length          | delta  | Kruskal-Wallis   | p=0.8596     | ε²=-0.055
  path_length          | theta  | Kruskal-Wallis   | p=0.1734     | ε²=0.049
  path_length          | alpha  | Kruskal-Wallis   | p=0.6894     | ε²=-0.041
  path_length          | beta   | Kruskal-Wallis   | p=0.2414     | ε²=0.027
  path_length          | gamma  | ANOVA            | p=0.1589     | η²=0.112
  sigma                | delta  | Kruskal-Wallis   | p=0.7546     | ε²=-0.046
  sigma                | theta  | ANOVA            | p=0.8782     | η²=0.008
  sigma                | alpha  | ANOVA            | p=0

## 5. Omnibus 检验 — ROI 连接性

9 ROI × 5 频段 = 45 检验 (within-ROI: 8 × 5 = 40, between-ROI: 1 × 5 = 5)

In [6]:
print('=' * 60)
print('ROI 连接性 Omnibus 检验')
print('=' * 60)

roi_results = []
if not roi_df.empty:
    roi_metrics = roi_df['roi_metric'].unique()
    for roi_metric in sorted(roi_metrics):
        for band in FREQ_BAND_KEYS:
            sub_df = roi_df[(roi_df['roi_metric'] == roi_metric) & (roi_df['freq_band'] == band)]
            if sub_df.empty:
                continue
            res = run_analysis_for_metric(sub_df, 'value', roi_metric, freq_band=band)
            if res:
                roi_results.append(res)
                sig_mark = '***' if res['p'] < 0.001 else '**' if res['p'] < 0.01 else '*' if res['p'] < 0.05 else ''
                print(f"  {roi_metric:<35s} | {band:<6s} | {res['test']:<16s} | "
                      f"p={res['p']:.4f} {sig_mark:>3s} | {res['es_name']}={res['effect_size']:.3f}")

print(f'\nROI 连接性检验总数: {len(roi_results)}')

ROI 连接性 Omnibus 检验
  between_prefrontal_left_pot         | delta  | Kruskal-Wallis   | p=0.7345     | ε²=-0.045
  between_prefrontal_left_pot         | theta  | ANOVA            | p=0.7892     | η²=0.015
  between_prefrontal_left_pot         | alpha  | ANOVA            | p=0.9201     | η²=0.005
  between_prefrontal_left_pot         | beta   | Kruskal-Wallis   | p=0.3511     | ε²=0.003
  between_prefrontal_left_pot         | gamma  | ANOVA            | p=0.1575     | η²=0.112
  within_central_L                    | delta  | ANOVA            | p=0.5775     | η²=0.035
  within_central_L                    | theta  | Kruskal-Wallis   | p=0.6526     | ε²=-0.037
  within_central_L                    | alpha  | Kruskal-Wallis   | p=0.3927     | ε²=-0.004
  within_central_L                    | beta   | Kruskal-Wallis   | p=0.1540     | ε²=0.056
  within_central_L                    | gamma  | ANOVA            | p=0.3587     | η²=0.064
  within_central_R                    | delta  | ANOVA    

## 6. FDR 校正

全脑连接性 + ROI 连接性合并 FDR 校正 (65 检验)

In [7]:
# 合并全脑 + ROI 结果做统一 FDR
all_conn_results = conn_results + roi_results
print(f'合并检验数: 全脑 {len(conn_results)} + ROI {len(roi_results)} = {len(all_conn_results)}')

conn_omnibus_df = apply_fdr(all_conn_results)

if not conn_omnibus_df.empty:
    print('\n连接性 Omnibus 结果 (FDR 校正后):')
    print(conn_omnibus_df[['metric', 'freq_band', 'test', 'p', 'p_fdr', 'significant', 'effect_size']].to_string(index=False))

# 分离全脑 / ROI 结果
conn_global_df = conn_omnibus_df[conn_omnibus_df['metric'].str.startswith('conn_')]
roi_omnibus_df = conn_omnibus_df[~conn_omnibus_df['metric'].str.startswith('conn_')]

# 汇总事后检验
posthoc_df = collect_posthoc(all_conn_results)

if not posthoc_df.empty:
    sig_posthoc = posthoc_df[posthoc_df['significant']]
    print(f'\n事后检验: {len(posthoc_df)} 对比较, {len(sig_posthoc)} 对显著')

# 保存 CSV
conn_omnibus_df.to_csv(OUTPUT_PATH / 'connectivity_omnibus_stats.csv', index=False, encoding='utf-8-sig')
if not roi_omnibus_df.empty:
    roi_omnibus_df.to_csv(OUTPUT_PATH / 'roi_omnibus_stats.csv', index=False, encoding='utf-8-sig')
if not posthoc_df.empty:
    posthoc_df.to_csv(OUTPUT_PATH / 'posthoc_stats.csv', index=False, encoding='utf-8-sig')
if not roi_df.empty:
    roi_df.to_csv(OUTPUT_PATH / 'roi_connectivity_data.csv', index=False, encoding='utf-8-sig')
print('\n✓ 统计结果已保存')

合并检验数: 全脑 20 + ROI 45 = 65

连接性 Omnibus 结果 (FDR 校正后):
                     metric freq_band           test        p    p_fdr  significant  effect_size
            conn_clustering     delta          ANOVA 0.582543 0.855759        False     0.034261
            conn_clustering     theta          ANOVA 0.194669 0.775281        False     0.100195
            conn_clustering     alpha          ANOVA 0.717584 0.855759        False     0.021183
            conn_clustering      beta          ANOVA 0.480874 0.822548        False     0.046137
            conn_clustering     gamma          ANOVA 0.601811 0.855759        False     0.032231
           conn_path_length     delta Kruskal-Wallis 0.859624 0.915993        False    -0.054757
           conn_path_length     theta Kruskal-Wallis 0.173409 0.775281        False     0.048523
           conn_path_length     alpha Kruskal-Wallis 0.689355 0.855759        False    -0.040516
           conn_path_length      beta Kruskal-Wallis 0.241398 0.775281   

## 7. 可视化

In [8]:
# 全脑连接性箱线图
if not conn_df.empty:
    for metric in CONN_METRICS:
        if metric not in conn_df.columns:
            continue
        for band in FREQ_BAND_KEYS:
            sub_df = conn_df[conn_df['freq_band'] == band] if 'freq_band' in conn_df.columns else conn_df
            if sub_df.empty:
                continue
            plot_boxplots(sub_df, metric,
                          f'{metric} ({band})', f'{metric}',
                          FIGURES_PATH / f'boxplot_conn_{metric}_{band}.png')
    print('✓ 全脑连接性箱线图已保存')

✓ 全脑连接性箱线图已保存


In [9]:
# ROI 连接性箱线图
if not roi_df.empty:
    roi_metrics = sorted(roi_df['roi_metric'].unique())
    for roi_metric in roi_metrics:
        for band in FREQ_BAND_KEYS:
            sub_df = roi_df[(roi_df['roi_metric'] == roi_metric) & (roi_df['freq_band'] == band)]
            if sub_df.empty:
                continue
            plot_boxplots(sub_df, 'value',
                          f'{roi_metric} ({band})', 'wPLI',
                          FIGURES_PATH / f'boxplot_roi_{roi_metric}_{band}.png')
    print('✓ ROI 连接性箱线图已保存')

✓ ROI 连接性箱线图已保存


In [10]:
# 效应量森林图
plot_effect_sizes(posthoc_df, FIGURES_PATH / 'effect_sizes_forest.png')

# 显著性热图 — 全脑连接性
plot_significance_heatmap(conn_global_df, '全脑连接性 Omnibus p 值 (FDR)',
                          FIGURES_PATH / 'heatmap_conn_significance.png')

# 显著性热图 — ROI 连接性
plot_significance_heatmap(roi_omnibus_df, 'ROI 连接性 Omnibus p 值 (FDR)',
                          FIGURES_PATH / 'heatmap_roi_significance.png')

print('✓ 可视化完成')

✓ 可视化完成


## 8. 生成报告

In [11]:
# 描述性统计
desc_sections = []

# 全脑连接性
desc_sections.append('### 全脑连接性指标\n')
if not conn_df.empty:
    for metric in CONN_METRICS:
        if metric not in conn_df.columns:
            continue
        for band in FREQ_BAND_KEYS:
            sub_df = conn_df[conn_df['freq_band'] == band] if 'freq_band' in conn_df.columns else conn_df
            if sub_df.empty:
                continue
            desc_sections.append(f'**{metric} ({band}):**\n')
            desc_sections.append(format_desc_table(sub_df, metric))

# ROI 连接性
desc_sections.append('### ROI 连接性指标\n')
if not roi_df.empty:
    roi_metrics = sorted(roi_df['roi_metric'].unique())
    for roi_metric in roi_metrics:
        for band in FREQ_BAND_KEYS:
            sub_df = roi_df[(roi_df['roi_metric'] == roi_metric) & (roi_df['freq_band'] == band)]
            if sub_df.empty:
                continue
            desc_sections.append(f'**{roi_metric} ({band}):**\n')
            desc_sections.append(format_desc_table(sub_df, 'value'))

desc_text = '\n'.join(desc_sections)

# Omnibus 表格
omnibus_parts = []
omnibus_parts.append(format_omnibus_table(conn_global_df, '全脑连接性'))
omnibus_parts.append(format_omnibus_table(roi_omnibus_df, 'ROI 连接性'))
omnibus_text = '\n'.join(omnibus_parts)

# 事后检验表格
posthoc_text = format_posthoc_table(posthoc_df)

# 样本量
sample_sizes = {g: conn_df[conn_df['group'] == g]['subject_id'].nunique() for g in GROUPS} if not conn_df.empty else {}

report = generate_report(
    title='连接性组间统计比较报告 (全脑 + ROI)',
    analysis_desc='全脑连接性 (4 指标 × 5 频段 = 20 检验) + ROI 连接性 (9 ROI × 5 频段 = 45 检验), 合计 65 检验统一 FDR 校正',
    sample_sizes=sample_sizes,
    desc_text=desc_text,
    omnibus_text=omnibus_text,
    posthoc_text=posthoc_text,
)

report_file = OUTPUT_PATH / 'connectivity_comparison_report.md'
with open(report_file, 'w', encoding='utf-8') as f:
    f.write(report)
print(f'✓ 报告已保存: {report_file}')

✓ 报告已保存: d:\LYW\REST_COM\reports\comparison\connectivity\connectivity_comparison_report.md


## 9. 分析完成

In [12]:
print('=' * 60)
print('连接性统计分析完成!')
print('=' * 60)

print(f'\n输出文件:')
for f in sorted(OUTPUT_PATH.glob('*')):
    if f.is_file():
        print(f'  - {f.name}')

print(f'\n可视化文件:')
for f in sorted(FIGURES_PATH.glob('*.png')):
    print(f'  - {f.name}')

# 显著结果摘要
print('\n' + '=' * 60)
print('显著结果摘要 (FDR < 0.05)')
print('=' * 60)

if not conn_omnibus_df.empty:
    sig_global = conn_global_df[conn_global_df['significant']] if not conn_global_df.empty else pd.DataFrame()
    sig_roi = roi_omnibus_df[roi_omnibus_df['significant']] if not roi_omnibus_df.empty else pd.DataFrame()

    print(f'\n全脑连接性: {len(sig_global)}/{len(conn_global_df)} 项显著')
    if len(sig_global) > 0:
        print(sig_global[['metric', 'freq_band', 'p_fdr', 'effect_size']].to_string(index=False))
    else:
        print('  无显著结果')

    print(f'\nROI 连接性: {len(sig_roi)}/{len(roi_omnibus_df)} 项显著')
    if len(sig_roi) > 0:
        print(sig_roi[['metric', 'freq_band', 'p_fdr', 'effect_size']].to_string(index=False))
    else:
        print('  无显著结果')

    if not posthoc_df.empty:
        sig_ph = posthoc_df[posthoc_df['significant']]
        print(f'\n显著事后比较: {len(sig_ph)}/{len(posthoc_df)} 对')
        if len(sig_ph) > 0:
            print(sig_ph[['metric', 'freq_band', 'pair', 'p', 'cohens_d']].to_string(index=False))

连接性统计分析完成!

输出文件:
  - connectivity_comparison_report.md
  - connectivity_omnibus_stats.csv
  - posthoc_stats.csv
  - roi_connectivity_data.csv
  - roi_omnibus_stats.csv

可视化文件:
  - boxplot_conn_avg_connectivity_alpha.png
  - boxplot_conn_avg_connectivity_beta.png
  - boxplot_conn_avg_connectivity_delta.png
  - boxplot_conn_avg_connectivity_gamma.png
  - boxplot_conn_avg_connectivity_theta.png
  - boxplot_conn_clustering_alpha.png
  - boxplot_conn_clustering_beta.png
  - boxplot_conn_clustering_delta.png
  - boxplot_conn_clustering_gamma.png
  - boxplot_conn_clustering_theta.png
  - boxplot_conn_path_length_alpha.png
  - boxplot_conn_path_length_beta.png
  - boxplot_conn_path_length_delta.png
  - boxplot_conn_path_length_gamma.png
  - boxplot_conn_path_length_theta.png
  - boxplot_conn_sigma_alpha.png
  - boxplot_conn_sigma_beta.png
  - boxplot_conn_sigma_delta.png
  - boxplot_conn_sigma_gamma.png
  - boxplot_conn_sigma_theta.png
  - boxplot_roi_between_prefrontal_left_pot_alpha.png
  -